In [1]:
# get eval dataset
# get retriever
# get context
# evaluate rank
# reranking
# evaluate rank

In [2]:
from pathlib import Path

import sys
sys.path.append("../")

from evaluation.eval import load_tests, evaluate_retrieval
from retrieval.faq_retriever import search_faq_hybrid
from retrieval.reranker import rerank
import pandas as pd

In [3]:
tests = load_tests()

In [4]:
def retrieving(test, top_k):
    context = search_faq_hybrid(test.question, top_k=top_k)
    nodes = getattr(context, "nodes", context)
    reranked_nodes = rerank(test.question, nodes, top_k)
    return nodes, reranked_nodes

In [5]:

rows = []
for index, test in enumerate(tests, 1):
    nodes, reranked_nodes = retrieving(test, 5)
    base_eval_result = evaluate_retrieval(test, nodes, 5)
    rerunked_eval_result = evaluate_retrieval(test, reranked_nodes, 5)
    rows.append({
        'question': base_eval_result['question'],
        'relevant_docs': base_eval_result['relevant_docs'],
        'retrieved_docs': base_eval_result['retrieved_docs'],
        'RERANKED_retrieved_docs': rerunked_eval_result['retrieved_docs'],
        'precision@5': base_eval_result['precision@5'],
        'RERANKED_precision@5': rerunked_eval_result['precision@5'],
        'recall@5': base_eval_result['recall@5'],
        'RERANKED_recall@5': rerunked_eval_result['recall@5'],
        'mrr': base_eval_result['mrr'],
        'RERANKED_mrr': rerunked_eval_result['mrr'],
        'ndcg@5': base_eval_result['ndcg@5'],
        'RERANKED_ndcg@5': rerunked_eval_result['ndcg@5'],
    })


report_df = pd.DataFrame(rows)
report_df

/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15188.27it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16297.85it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids |

,question,relevant_docs,retrieved_docs,RERANKED_retrieved_docs,precision@5,RERANKED_precision@5,recall@5,RERANKED_recall@5,mrr,RERANKED_mrr,ndcg@5,RERANKED_ndcg@5
0,How do I register for a HopShop account using ...,[1],"[1, 2, 6, 4, 49]","[1, 2, 49, 4, 6]",0.20,0.20,1.0,1.0,1.000000,1.000000,1.000000,1.00000
1,How can I sign up using my Google or Facebook ...,[2],"[2, 6, 1, 3, 75]","[2, 3, 1, 6, 75]",0.20,0.20,1.0,1.0,1.000000,1.000000,1.000000,1.00000
2,How can I enable signing in with my email addr...,[3],"[3, 1, 2, 51, 49]","[3, 1, 2, 49, 51]",0.20,0.20,1.0,1.0,1.000000,1.000000,1.000000,1.00000
3,How do I register for a HopShop account using ...,[4],"[4, 1, 2, 6]","[4, 1, 2, 6]",0.25,0.25,1.0,1.0,1.000000,1.000000,1.000000,1.00000
4,How can I download my personal data from my Ho...,[5],"[5, 7, 8, 6, 11]","[5, 11, 6, 7, 8]",0.20,0.20,1.0,1.0,1.000000,1.000000,1.000000,1.00000
5,Am I allowed to have multiple HopShop accounts...,[6],"[6, 50, 40, 46, 17]","[6, 40, 46, 17, 50]",0.20,0.20,1.0,1.0,1.000000,1.000000,1.000000,1.00000
6,How do I close my HopShop account?,[7],"[7, 16, 6, 14, 2]","[7, 16, 14, 6, 2]",0.20,0.20,1.0,1.0,1.000000,1.000000,1.000000,1.00000
7,How can I withdraw from my agreement with HopS...,[8],"[9, 8, 10, 7, 14]","[9, 7, 8, 14, 10]",0.20,0.20,1.0,1.0,0.500000,0.333333,0.630930,0.50000
8,Under what conditions can I withdraw from my a...,[9],"[9, 8, 10, 12, 13]","[9, 8, 12, 13, 10]",0.20,0.20,1.0,1.0,1.000000,1.000000,1.000000,1.00000
9,How can I withdraw from my agreement and what ...,[10],"[10, 9, 15, 11, 13]","[10, 9, 11, 13, 15]",0.20,0.20,1.0,1.0,1.000000,1.000000,1.000000,1.00000


In [6]:
def _stringify_markdown_value(value):
    if value is None:
        return "-"
    if isinstance(value, float):
        return f"{value:.3f}"
    if isinstance(value, list):
        return ", ".join(str(item) for item in value) if value else "-"
    return str(value)


def _to_markdown_table(df):
    columns = list(df.columns)
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join("---" for _ in columns) + " |"
    rows = []

    for record in df.to_dict(orient="records"):
        rendered = [_stringify_markdown_value(record.get(column)) for column in columns]
        rows.append("| " + " | ".join(rendered) + " |")

    return "\n".join([header, separator, *rows])


def build_reranking_markdown_report(report_df, output_path="../reports/faq_reranking_evaluation.md"):
    summary_df = pd.DataFrame([
        {
            "questions": int(len(report_df)),
            "precision@5": round(float(report_df["precision@5"].mean()), 3),
            "RERANKED_precision@5": round(float(report_df["RERANKED_precision@5"].mean()), 3),
            "recall@5": round(float(report_df["recall@5"].mean()), 3),
            "RERANKED_recall@5": round(float(report_df["RERANKED_recall@5"].mean()), 3),
            "mrr": round(float(report_df["mrr"].mean()), 3),
            "RERANKED_mrr": round(float(report_df["RERANKED_mrr"].mean()), 3),
            "ndcg@5": round(float(report_df["ndcg@5"].mean()), 3),
            "RERANKED_ndcg@5": round(float(report_df["RERANKED_ndcg@5"].mean()), 3),
        }
    ])

    lines = [
        "# FAQ Reranking Evaluation Report",
        "",
        "Source notebook: `notebooks/09_faq_reranking.ipynb`",
        "",
        "## Summary",
        "",
        f"- Evaluated questions: {int(len(report_df))}",
        f"- Mean precision@5: {summary_df.iloc[0]['precision@5']:.3f} -> {summary_df.iloc[0]['RERANKED_precision@5']:.3f}",
        f"- Mean recall@5: {summary_df.iloc[0]['recall@5']:.3f} -> {summary_df.iloc[0]['RERANKED_recall@5']:.3f}",
        f"- Mean MRR: {summary_df.iloc[0]['mrr']:.3f} -> {summary_df.iloc[0]['RERANKED_mrr']:.3f}",
        f"- Mean nDCG@5: {summary_df.iloc[0]['ndcg@5']:.3f} -> {summary_df.iloc[0]['RERANKED_ndcg@5']:.3f}",
        "",
        "## Final Metrics",
        "",
        _to_markdown_table(summary_df),
        "",
        "## Per-question Results",
        "",
    ]

    for index, row in report_df.iterrows():
        metrics_df = pd.DataFrame([
            {
                "precision@5": row['precision@5'],
                "RERANKED_precision@5": row['RERANKED_precision@5'],
                "recall@5": row['recall@5'],
                "RERANKED_recall@5": row['RERANKED_recall@5'],
                "mrr": row['mrr'],
                "RERANKED_mrr": row['RERANKED_mrr'],
                "ndcg@5": row['ndcg@5'],
                "RERANKED_ndcg@5": row['RERANKED_ndcg@5'],
            }
        ])

        lines.extend([
            f"### Question {index + 1}",
            "",
            f"- Query: {row['question']}",
            f"- Relevant docs: {row['relevant_docs']}",
            f"- Retrieved docs: {row['retrieved_docs']}",
            f"- Reranked docs: {row['RERANKED_retrieved_docs']}",
            "",
            _to_markdown_table(metrics_df),
            "",
        ])

    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return output, summary_df


In [7]:
report_path, summary_df = build_reranking_markdown_report(report_df)
print(f"Markdown report saved to: {report_path}")
summary_df


Markdown report saved to: ../reports/faq_reranking_evaluation.md


,questions,precision@5,RERANKED_precision@5,recall@5,RERANKED_recall@5,mrr,RERANKED_mrr,ndcg@5,RERANKED_ndcg@5
0,50,0.198,0.198,0.98,0.98,0.857,0.91,0.889,0.928
